In [1]:
import polars as pl

In [20]:
df = pl.read_parquet("../../data/ranked_significant_results.parquet")

In [21]:
def process_row(row):
    col_names = row["col_name"]
    values = row["col_value"]
    attributes = [f"{col}={val}" for col, val in zip(col_names, values)]
    results = f"Samples from the project {row['bioproject']} with the following attributes: {', '.join(attributes)} had a mean concentration of totiviridiae of {row['mean_positive']} and a mean concentration of totiviridiae of {row['mean_negative']} in the negative samples."
    return results

# add a new column to the dataframe with the processed results
df = df.with_columns(
    pl.struct(["col_name", "col_value", "mean_positive", "mean_negative", "bioproject"])
    .map_elements(process_row, return_dtype=pl.String)
    .alias("processed_result")
)

In [22]:
# stack all processed_result into a big string
all_results = "\n".join(df["processed_result"].to_list())

In [23]:
# save this to a txt
with open("../../data/all_hypotheses.txt", "w", encoding="utf-8") as f:
    f.write(all_results)

In [24]:
from openai import OpenAI
from openai.providers import bedrock
from dotenv import load_dotenv
import asyncio
import os
load_dotenv()

# retries with exponential backoff on rate limits and transient errors
client = OpenAI(max_retries=5)

In [31]:
prompt = f"""
Analyzing SRA metadata against totiviridae abundance, we found the following statistical findings, after error correction. Please analyze these and come up with things that seem to be related across bioprojects.\n
{all_results}"""

MODEL = "openai.gpt-6.1-sol"

# price in $ per 1M tokens
INPUT = 2.20
OUTPUT = 11.00

def query_hypothesis(prompt):
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}]
        )
    except Exception as e:
        # leave the row empty so it is retried on the next run
        print(f"request failed: {e}")
        return {"response": "", "input_tokens": 0, "output_tokens": 0, "cost": 0.0}
    input_tokens = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens
    cost = (output_tokens * OUTPUT + input_tokens * INPUT) / 1000000
    return {
        "response": response.choices[0].message.content or "",
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "cost": cost,
    }

response = query_hypothesis(prompt)

In [34]:
response["response"]

# write response to a txt file
with open("../../data/hypothesis_all_results.txt", "w", encoding="utf-8") as f:
    f.write(response["response"])

In [36]:
response["cost"]

0.1677258